# HeatWatch NYC: exploring the data before building the pipeline

Do apartment buildings that use energy poorly also leave their tenants cold?

This notebook is where I figure out whether the data can even answer that. Two NYC Open Data sources:

- **311 service requests** (`erm2-nwe9`): heat complaints from tenants, published daily
- **LL84 energy benchmarking** (`5zyy-y8am`): yearly energy scores for large buildings

They join on BBL (borough-block-lot), the city's tax lot ID.

Nothing gets built in AWS until I know how both datasets actually behave.

## Why 311 and not just LL84

My first plan was a daily pipeline on LL84 alone. Then I checked the dataset's metadata page: data change frequency is *annually*, automation is *no*. A daily Lambda would come back empty almost every day.

So the roles split:

- 311 heat complaints change every day, so they become the **fact table** the pipeline loads incrementally
- LL84 changes once a year, so it becomes the **building dimension**

**My prediction:** buildings with low Energy Star scores will have more heat complaints, because inefficient heating systems and poor insulation make it harder to keep apartments warm.

In [1]:
import os
import requests
import pandas as pd

# token is optional while exploring - the API works without one, just with a lower rate limit.
# once it's a Codespaces secret, os.getenv finds it automatically, no code changes needed
APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

URL_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"    # daily: complaints
URL_LL84 = "https://data.cityofnewyork.us/resource/5zyy-y8am.json"  # yearly: building energy

## First look: five heat complaints

Just the five newest, transposed so every field fits on screen. I mainly want to know whether `bbl` is actually filled in, since the whole join depends on it.

In [2]:
# only heat complaints - potholes and noise are fun, but not what this question is about
params = {
    "complaint_type": "HEAT/HOT WATER",   # Socrata lets you filter just by naming the column
    "$order": "created_date DESC",
    "$limit": 5,
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=30)
resp.raise_for_status()   # fail loudly on a bad response instead of treating an error page as data
heat = pd.DataFrame(resp.json())
heat.T   # transposed: 5 records become columns, so all ~40 fields fit on screen

,0,1,2,3,4
unique_key,70518877,70515926,70518876,70514467,70523251
created_date,2026-09-23T23:54:22.000,2026-09-23T23:51:04.000,2026-09-23T23:48:10.000,2026-09-23T23:48:01.000,2026-09-23T23:42:19.000
agency,HPD,HPD,HPD,HPD,HPD
agency_name,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...,Department of Housing Preservation and Develop...
complaint_type,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER,HEAT/HOT WATER
descriptor,ENTIRE BUILDING,ENTIRE BUILDING,ENTIRE BUILDING,ENTIRE BUILDING,ENTIRE BUILDING
descriptor_2,NO HOT WATER,NO HOT WATER,NO HOT WATER,NO HOT WATER,NO HOT WATER
location_type,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING,RESIDENTIAL BUILDING
incident_zip,10019,11210,11216,10032,10032
incident_address,358 WEST 51 STREET,2325 FOSTER AVENUE,30 ROGERS AVENUE,76 ST NICHOLAS PLACE,76 ST NICHOLAS PLACE


What I noticed:

- `bbl` is filled in for all five, and the first digit matches the borough (1 = Manhattan, 2 = Bronx, 3 = Brooklyn, 4 = Queens). Good sign.
- Every complaint is from HPD and tagged `RESIDENTIAL BUILDING`, so these really are tenants.
- One complaint is marked as a duplicate of a building-wide complaint. Ten tenants calling about one broken boiler is one problem, not ten. I'll need to decide how to count those.
- The newest record was about a day and a half old when I pulled it. The city publishes in batches, not live, so the pipeline has to expect a lag.
- All five are still `Open`. They'll get updated later, which is why the incremental load should key on `:updated_at`, not `created_date`.

## Does the data actually change daily?

Five rows show the shape. To trust any percentage I need more, so: every heat complaint from the last 30 days.

In [3]:
# 5 rows showed the shape - now I need enough rows to actually trust a percentage.
# last 30 days: big enough to count, small enough to be polite to the API
params = {
    "$select": ":updated_at, unique_key, created_date, descriptor, descriptor_2, status, bbl",
    "$where": "complaint_type = 'HEAT/HOT WATER' AND created_date > '2026-08-25T00:00:00'",
    "$limit": 50000,
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=60)
resp.raise_for_status()
recent = pd.DataFrame(resp.json())
print(f"{len(recent):,} heat/hot water complaints in the last 30 days")

3,986 heat/hot water complaints in the last 30 days


In [4]:
# the hot water question - answered by data instead of by guessing
recent["descriptor_2"].value_counts(dropna=False)

descriptor_2
NO HOT WATER         3853
HEAT ON IN SUMMER     133
Name: count, dtype: int64

3,759 complaints in 30 days, about 125 a day. But look at what they are:

| Complaint | Count |
|---|---|
| NO HOT WATER | 3,632 |
| HEAT ON IN SUMMER | 127 |

Zero "no heat" complaints. Makes sense in September, when nobody's heat is on yet. It also means the last 30 days can't test my hypothesis at all.

(Side note: "heat on in summer" means heat running when it shouldn't be. Could say something about building controls and wasted energy. Parking that for later.)

99.8% of complaints have a BBL, so the join key itself is reliable.

In [5]:

# what share of complaints can actually join to an LL84 building?
recent["bbl"].notna().mean()

np.float64(0.997491219267436)

## Checking last winter instead

If September can't test a heating hypothesis, January can. `$group` makes the API do the counting, so I get back three rows instead of downloading tens of thousands.

In [6]:
# September is the wrong month to test a heating hypothesis - so check last January instead.
# $group makes Socrata count server-side: we get back a few rows, not thousands
params = {
    "$select": "descriptor_2, count(*) AS n",
    "$where": "complaint_type = 'HEAT/HOT WATER' "
              "AND created_date between '2026-01-01T00:00:00' and '2026-01-31T23:59:59'",
    "$group": "descriptor_2",
    "$order": "n DESC",
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=60)
resp.raise_for_status()
pd.DataFrame(resp.json())

,descriptor_2,n
0,NO HEAT,50604
1,NO HEAT AND NO HOT WATER,23444
2,NO HOT WATER,5880


January 2026 is a completely different dataset:

| Complaint | Count | Share |
|---|---|---|
| NO HEAT | 50,604 | 63% |
| NO HEAT AND NO HOT WATER | 23,444 | 29% |
| NO HOT WATER | 5,880 | 7% |

About 2,600 complaints a day, roughly 20x September, and 92% involve missing heat.

**Decision: filter to `NO HEAT` and `NO HEAT AND NO HOT WATER`.** Both mean the tenant is cold. "No hot water" alone is usually a water heater failing, which doesn't test anything about insulation or heating efficiency.

**Design change: backfill, then go incremental.** If the pipeline only loads daily from today, there's nothing worth analyzing until January. So it loads last winter once, then picks up new data each day.

## Which years of LL84 exist?

A complaint should be compared to the building's score *as it was at the time*. So I need to know which reporting years are available.

In [7]:
# which reporting years does LL84 actually have?
# matters because a Jan 2026 complaint should be compared to the building's score *at that time*
params = {
    "$select": "report_year, count(*) AS n",
    "$group": "report_year",
    "$order": "report_year",
}
resp = requests.get(URL_LL84, headers=HEADERS, params=params, timeout=60)
resp.raise_for_status()
pd.DataFrame(resp.json())

,report_year,n
0,2022,30485
1,2023,33684
2,2024,39090


2022, 2023, and 2024, with more buildings each year (30K → 34K → 39K).

No 2025 yet, since buildings report the previous year and the city publishes later. So a January 2026 complaint gets compared to the 2024 score, the latest that existed then. Using a later score would be like judging an old stock pick with next year's prices.

## How many cold buildings are actually in LL84?

LL84 only covers large buildings. Heat complaints come from buildings of every size. So the real question is how much of the complaint data survives the join.

My guess before running it: the share of *complaints* matched will be higher than the share of *buildings* matched.

In [8]:
# 2024 = latest year that existed last winter, so it's the fair comparison
cols = [
    "nyc_borough_block_and_lot", "property_name", "primary_property_type",
    "year_built", "energy_star_score", "site_eui_kbtu_ft", "property_gfa_self_reported",
]
params = {"$select": ", ".join(cols), "report_year": "2024", "$limit": 50000}
resp = requests.get(URL_LL84, headers=HEADERS, params=params, timeout=120)
resp.raise_for_status()
ll84 = pd.DataFrame(resp.json())

# grain check - if unique BBLs < rows, some lots have multiple properties reported
print(f"{len(ll84):,} rows, {ll84['nyc_borough_block_and_lot'].nunique():,} unique BBLs")


39,090 rows, 27,922 unique BBLs


In [9]:
params = {
    "$select": "bbl, count(*) AS complaints",
    "$where": "complaint_type = 'HEAT/HOT WATER' "
              "AND descriptor_2 in ('NO HEAT', 'NO HEAT AND NO HOT WATER') "
              "AND created_date between '2026-01-01T00:00:00' and '2026-01-31T23:59:59'",
    "$group": "bbl",
    "$limit": 50000,
}
resp = requests.get(URL_311, headers=HEADERS, params=params, timeout=120)
resp.raise_for_status()
jan = pd.DataFrame(resp.json())
jan["complaints"] = jan["complaints"].astype(int)   # API sends numbers as text, remember
print(f"{len(jan):,} buildings had heat complaints in Jan 2026")

16,778 buildings had heat complaints in Jan 2026


In [10]:
# dedupe first - a duplicated BBL in ll84 would silently double-count complaints in the merge
ll84_one = ll84.drop_duplicates("nyc_borough_block_and_lot")
matched = jan.merge(ll84_one, left_on="bbl", right_on="nyc_borough_block_and_lot",
                    how="left", indicator=True)   # indicator adds a column saying where each row came from

in_ll84 = matched["_merge"] == "both"
print(f"buildings matched:  {in_ll84.mean():.1%}")
print(f"complaints matched: {matched.loc[in_ll84, 'complaints'].sum() / matched['complaints'].sum():.1%}")

buildings matched:  40.6%
complaints matched: 58.7%


- 16,778 buildings had heat complaints in January 2026
- 40.6% of them matched an LL84 building
- but those buildings account for 58.7% of complaints

Prediction held. The reason: LL84 is big buildings, big buildings have more apartments, more apartments means more people to call. So the buildings that match are the ones generating the most complaints each.

Also this, from the grain check:

> 39,090 rows, 27,922 unique BBLs

About 11,000 rows share a BBL with another row. `drop_duplicates` just kept whichever came first, which could be quietly picking random scores. Need to look before trusting anything.

## What's going on with the duplicate BBLs?

One tax lot can hold several buildings: a hospital campus, a co-op with multiple towers, a big housing development. That part I expected. What I didn't expect is below.

In [11]:
# keep=False marks ALL copies of a duplicated BBL, not just the extras
dupes = ll84[ll84.duplicated("nyc_borough_block_and_lot", keep=False)]
print(f"{dupes['nyc_borough_block_and_lot'].nunique():,} BBLs have more than one row")
dupes.sort_values("nyc_borough_block_and_lot").head(12)

3,568 BBLs have more than one row


,nyc_borough_block_and_lot,property_name,primary_property_type,year_built,energy_star_score,site_eui_kbtu_ft,property_gfa_self_reported
18194,0000000000,Sample Engineering Laboratory (US),Laboratory,1901,Not Available,Not Available,100000
27565,0000000000,Multifamily Building,Multifamily Housing,2000,Not Available,Not Available,20000
18192,"0000000001, 0000000002",Sample Office (US),Office,1975,Not Available,Not Available,275000
28789,"0000000001, 0000000002",Child 1,Residence Hall/Dormitory,2000,Not Available,Not Available,50000
16718,1-00545-0026,10 Astor Place,Office,1910,Not Available,74.8,119434
16719,1-00545-0026,440 Lafayette,Office,1910,Not Available,46.5,86260
17246,1-00545-0026,Copy of 740 Broadway,Office,1910,51,106.8,137261
32531,1-01081-0080,537 WEST 52 STREET,Office,1907,Not Available,Not Available,36351
32532,1-01081-0080,549 WEST 52 STREET,Other,1907,Not Available,Not Available,21940
11999,1-01851-0008,135 West 96th Street,Multifamily Housing,1968,5,76.2,504485


The BBL column comes in at least six formats:

| Format | Example | Rows |
|---|---|---|
| Clean 10 digits (what 311 uses) | `1020690001` | 37,077 |
| Dashed | `1-00545-0026` | 887 |
| Comma list | `0000000001, 0000000002` | 15 |
| Semicolon list | `3043930001;3043710001` | ~880 |
| Missing | `Not Available` | 219 |
| EPA demo records | `0000000000`, "Sample Office (US)" | 8 |

The dashed ones are the sneaky problem. `1-00545-0026` and `1005450026` are the same lot, but as strings they never match, so every dashed BBL silently failed to join.

I guessed around 10,000 rows would be dashed. It was 887. Glad I counted instead of assuming.

Also noticed a record named "Copy of 740 Broadway" sharing a lot with the real buildings. Might be a duplicated entry that double-counts floor area. Worth a filter later.

In [12]:
bbl = ll84["nyc_borough_block_and_lot"].astype(str)
print("clean 10-digit :", bbl.str.fullmatch(r"\d{10}").sum())
print("dashed         :", bbl.str.contains("-").sum())
print("comma lists    :", bbl.str.contains(",").sum())
print("all zeros/fake :", bbl.str.startswith("0").sum())

clean 10-digit : 37077
dashed         : 887
comma lists    : 15
all zeros/fake : 8


In [13]:
# anything that isn't clean, dashed, a list, or fake - what's left?
known = (bbl.str.fullmatch(r"\d{10}") | bbl.str.contains("-|,") | bbl.str.startswith("0"))
other = bbl[~known]
print(f"{len(other):,} rows in an unknown format")
other.value_counts().head(15)

1,112 rows in an unknown format


nyc_borough_block_and_lot
Not Available                                                                   219
3035590001;3035740001;3035750011;3035870001;3036010026;3035880001;3035730001     99
3043930001;3043710001;3043970001;3043750050                                      17
1020370011;1020370001;1020160060                                                 10
1018360001;1018550001                                                             3
1021390017;1021390030                                                             2
3035440001;3035610001                                                             2
3037270001;3037450001                                                             2
3005380001;3005330001;3005570001                                                  2
3073870001;3074051001;3073890001;3074080001                                       2
4101250033;4101460051;4101250012;4101250063;4101270001;4101480001;4121480100      2
2055640001;2055670001;2055820001                  

## Normalizing BBLs

One function that handles every format above and returns a *list*, because some properties span several lots. Tested on known examples before I trust it.

In [14]:
import re

def clean_bbl(raw):
    """Turn LL84's messy BBL field into a list of clean 10-digit BBLs (the format 311 uses).

    Formats found in the wild: '1020690001', '1-00545-0026', comma lists, semicolon lists,
    'Not Available', and EPA demo records like '0000000000'.
    """
    cleaned = []
    for piece in re.split(r"[;,]", str(raw)):   # both separators show up, because of course they do
        piece = piece.strip()
        if "-" in piece:
            parts = piece.split("-")
            if len(parts) != 3:
                continue   # a dash format I haven't seen - skip it rather than guess
            boro, block, lot = parts
            piece = boro + block.zfill(5) + lot.zfill(4)   # zfill guards against dropped leading zeros
        # valid = 10 digits with a real borough (1-5). This one rule also drops
        # 'Not Available' and the 0000... demo records - no special cases needed
        if re.fullmatch(r"[1-5]\d{9}", piece):
            cleaned.append(piece)
    return cleaned
    

In [15]:
for test in ["1020690001", "1-00545-0026", "0000000001, 0000000002",
             "3043930001;3043710001", "Not Available"]:
    print(f"{test!r:30} -> {clean_bbl(test)}")

'1020690001'                   -> ['1020690001']
'1-00545-0026'                 -> ['1005450026']
'0000000001, 0000000002'       -> []
'3043930001;3043710001'        -> ['3043930001', '3043710001']
'Not Available'                -> []


In [16]:
ll84["bbl_list"] = ll84["nyc_borough_block_and_lot"].apply(clean_bbl)
# explode: one row per BBL, so a campus spanning 7 lots can match a complaint on any of them
ll84_long = ll84.explode("bbl_list").dropna(subset=["bbl_list"])
print(f"{len(ll84):,} rows -> {len(ll84_long):,} after splitting lists, "
      f"{ll84_long['bbl_list'].nunique():,} unique BBLs")

# only need "is this BBL in LL84 at all?" here, so dedupe the key column before joining
lookup = ll84_long[["bbl_list"]].drop_duplicates()
matched = jan.merge(lookup, left_on="bbl", right_on="bbl_list", how="left", indicator=True)
in_ll84 = matched["_merge"] == "both"
print(f"buildings matched:  {in_ll84.mean():.1%}   (was 40.6%)")
print(f"complaints matched: {matched.loc[in_ll84, 'complaints'].sum() / matched['complaints'].sum():.1%}   (was 58.7%)")


39,090 rows -> 40,941 after splitting lists, 27,424 unique BBLs
buildings matched:  42.5%   (was 40.6%)
complaints matched: 61.1%   (was 58.7%)


All five test cases came back right, and the re-match:

| | Before | After cleaning |
|---|---|---|
| Buildings matched | 40.6% | 42.5% |
| Complaints matched | 58.7% | 61.1% |

About 320 more buildings and roughly 1,800 more January heat complaints now connect to building data. A naive join would have dropped them without any error.

Unique BBLs went *down* (27,922 → 27,424). That's right: dashed and clean versions of the same lot were being counted twice before.

**Scope:** after cleaning, 42.5% of buildings with January heat complaints match LL84, and they account for 61.1% of complaints. This analysis is about large buildings.

These five test cases should become dbt tests in the pipeline.

## First test of the hypothesis

Three choices before running anything:

1. **Apartment buildings only.** Office towers never get HPD heat complaints. Keeping them would add thousands of fake zeros.
2. **Weight by floor area.** When several buildings share a lot, the bigger one counts more toward the lot's score, same idea as credits in a GPA. A campus spread across 7 lots gets its floor area split evenly, so it isn't counted 7 times. (Even split is an assumption, since I don't know which building sits on which lot.)
3. **Keep the zeros.** Buildings nobody complained about are the control group. Only looking at buildings with complaints would be like studying only sick patients.

Complaints are normalized per 100,000 sq ft, so a 500-unit tower doesn't look worse just because more people live there.

In [17]:
# heat complaints come from tenants - offices and warehouses would just add fake zeros
res = ll84[ll84["primary_property_type"] == "Multifamily Housing"].copy()
res["n_lots"] = res["bbl_list"].str.len()   # count lots BEFORE exploding
res = res.explode("bbl_list").dropna(subset=["bbl_list"])

# text -> numbers; "Not Available" quietly becomes NaN, which is what we want
for col in ["energy_star_score", "property_gfa_self_reported"]:
    res[col] = pd.to_numeric(res[col], errors="coerce")

# a 7-lot campus got copied onto 7 rows with its FULL floor area each - split it evenly.
# even split is an assumption: I don't know which building sits on which lot
res["gfa_share"] = res["property_gfa_self_reported"] / res["n_lots"]
print(f"{res['bbl_list'].nunique():,} residential lots")

18,693 residential lots


In [18]:
# GPA-style: score weighted by floor area, using only buildings that actually have a score
scored = res.dropna(subset=["energy_star_score", "gfa_share"])
scored = scored.assign(score_x_area=scored["energy_star_score"] * scored["gfa_share"])
lot = scored.groupby("bbl_list").agg(score_x_area=("score_x_area", "sum"),
                                     scored_area=("gfa_share", "sum"))
lot["energy_star"] = lot["score_x_area"] / lot["scored_area"]

# total floor area includes unscored buildings too - their tenants can still complain
lot = lot.join(res.groupby("bbl_list")["gfa_share"].sum().rename("total_gfa"))
lot = lot[lot["total_gfa"] > 0]
print(f"{len(lot):,} lots with a usable score and floor area")

16,719 lots with a usable score and floor area


18,693 residential lots, and 16,719 of them (89%) have a usable score and floor area.

About 11% have no Energy Star score and drop out. If the worst-run buildings are also the ones that don't report properly, that could bias things. Noting it.

## Control 1: building size

Four equal groups of lots by floor area. If the complaint rate drops steadily as buildings get bigger, size is part of the story, and the Energy Star comparison has to happen *within* each size group to be fair.

In [19]:
# LEFT join from buildings: lots nobody complained about stay in, with 0 - that's the control group
df = lot.join(jan.set_index("bbl")["complaints"], how="left").fillna({"complaints": 0})
df["band"] = pd.cut(df["energy_star"], bins=[0, 25, 50, 75, 100],
                    labels=["1-25", "26-50", "51-75", "76-100"], include_lowest=True)

summary = df.groupby("band", observed=True).agg(
    lots=("complaints", "size"),
    pct_with_complaint=("complaints", lambda s: (s > 0).mean()),
    complaints=("complaints", "sum"),
    gfa=("total_gfa", "sum"),
)
# pooled rate: total complaints / total floor area, so one huge building can't dominate
summary["per_100k_sqft"] = summary["complaints"] / summary["gfa"] * 100_000
summary.round(3)

,lots,pct_with_complaint,complaints,gfa,per_100k_sqft
band,,,,,
1-25,2464,0.374,6317.0,4.579884e+08,1.379
26-50,3099,0.442,9578.0,4.811410e+08,1.991
51-75,4546,0.403,11324.0,6.027488e+08,1.879
76-100,6610,0.355,14905.0,8.048620e+08,1.852


**Size result:** [write what the per_100k_sqft column does from smallest to largest]

**What it means for the hypothesis:** [does size explain the pattern, part of it, or none of it?]

In [20]:
# does size alone explain complaint rates? qcut makes 4 groups with equal numbers of lots
df["size_band"] = pd.qcut(df["total_gfa"], 4,
                          labels=["smallest 25%", "small-mid", "mid-large", "largest 25%"])
size = df.groupby("size_band", observed=True).agg(
    lots=("complaints", "size"),
    complaints=("complaints", "sum"),
    gfa=("total_gfa", "sum"),
    avg_score=("energy_star", "mean"),
)
size["per_100k_sqft"] = size["complaints"] / size["gfa"] * 100_000
size.round(2)

,lots,complaints,gfa,avg_score,per_100k_sqft
size_band,,,,,
smallest 25%,4181,7976.0,1.378864e+08,58.05,5.78
small-mid,4179,9981.0,2.289842e+08,62.95,4.36
mid-large,4179,12149.0,3.815595e+08,66.59,3.18
largest 25%,4180,12018.0,1.598310e+09,59.79,0.75


## What this notebook means for the pipeline

Everything here turns into a requirement for the build:

**Ingestion (Lambda)**
- Filter 311 to `HEAT/HOT WATER`; keep `descriptor_2` so the heat vs. hot-water split can happen in dbt
- Watermark on `:updated_at`, not `created_date`, because complaints get updated after they're filed
- Expect a publishing lag of about a day
- One-time backfill of last winter, then daily incremental loads

**Staging (dbt)**
- Cast everything, since the API sends numbers as text
- Turn `"Not Available"` into real nulls
- Normalize BBLs with the same logic as `clean_bbl`, and turn those five test cases into dbt tests
- Drop EPA demo records

**Modeling (dbt)**
- 311 complaints = fact table, LL84 buildings = dimension
- Snapshot LL84 yearly (SCD2) so each complaint joins to the score that existed at the time
- Aggregate multi-building lots by floor area, not by picking one row

In [21]:
# does size alone explain complaint rates? qcut makes 4 groups with equal numbers of lots
df["size_band"] = pd.qcut(df["total_gfa"], 4,
                          labels=["smallest 25%", "small-mid", "mid-large", "largest 25%"])
size = df.groupby("size_band", observed=True).agg(
    lots=("complaints", "size"),
    complaints=("complaints", "sum"),
    gfa=("total_gfa", "sum"),
    avg_score=("energy_star", "mean"),
)
size["per_100k_sqft"] = size["complaints"] / size["gfa"] * 100_000
size.round(2)

,lots,complaints,gfa,avg_score,per_100k_sqft
size_band,,,,,
smallest 25%,4181,7976.0,1.378864e+08,58.05,5.78
small-mid,4179,9981.0,2.289842e+08,62.95,4.36
mid-large,4179,12149.0,3.815595e+08,66.59,3.18
largest 25%,4180,12018.0,1.598310e+09,59.79,0.75


In [22]:
# the fair test: within each size group, does the complaint rate change with Energy Star score?
within = df.groupby(["size_band", "band"], observed=True).agg(
    lots=("complaints", "size"),
    complaints=("complaints", "sum"),
    gfa=("total_gfa", "sum"),
)
within["per_100k_sqft"] = within["complaints"] / within["gfa"] * 100_000

# rows = size groups, columns = Energy Star bands; read each row left to right
within["per_100k_sqft"].unstack("band").round(2)

band,1-25,26-50,51-75,76-100
size_band,,,,
smallest 25%,7.50,7.67,5.54,3.82
small-mid,5.17,4.88,4.77,3.58
mid-large,3.37,4.28,3.08,2.84
largest 25%,0.41,0.78,0.76,0.98


In [23]:
within["lots"].unstack("band")

band,1-25,26-50,51-75,76-100
size_band,,,,
smallest 25%,793,906,998,1484
small-mid,557,756,1171,1695
mid-large,418,644,1216,1901
largest 25%,696,793,1161,1530


**Size result:** Complaint rate drops sharply with size, from 5.78 per 100K sq ft in the smallest quarter
to 0.75 in the largest, about an 8x gap. Size matters far more than Energy Star score in the raw data.

**Within each size group, the picture changes.** In the smallest three quarters of buildings,
better Energy Star scores go with fewer complaints (about 49%, 31%, and 16% lower from the worst band
to the best). In the largest quarter, it reverses.

The largest buildings hold about two-thirds of all floor area, so they dominated the pooled
comparison and hid the pattern everywhere else. Simpson's paradox, in my own data.

**Where the hypothesis stands:** consistent with the data for most buildings once size is controlled,
not yet tested for significance, and building age is still to check.